<a href="https://colab.research.google.com/github/PariAgrawal1234/Sequence_completion_LSTM_model/blob/main/Sequence_completion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/qoute_dataset.csv")

In [ ]:
df.info()
df.isna().sum()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
quote = df['quote']
quote = quote.str.lower()

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)
quote = quote.apply(lambda x: x.translate(translator))
quote

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...
...,...
3033,the past beats inside me like a second heart
3034,damn claire warn a guy before you do a facepla...
3035,can you be a girl for a few secondsim always a...
3036,thats what fiction is for its for getting at t...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 10000
tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quote)
word_indx = tokenizer.word_index
print(len(word_indx))
vocab_size = len(word_indx) + 1

8978


In [ ]:
seq = tokenizer.texts_to_sequences(quote)
print(seq[0:2])
print(quote[0:2])

[[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461], [947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]]
0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
Name: quote, dtype: object


In [ ]:
X = []
y = []

for sq in seq:
  for i in range(1, len(sq)):
    input_sq = sq[:i]
    output_sq = sq[i]
    X.append(input_sq)
    y.append(output_sq)

In [ ]:
max_len = max(len(x) for x in X)
max_len

745

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

X_padded = np.array(X_padded)
y = np.array(y)

In [ ]:
y.shape

(85271,)

In [ ]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes = vocab_size)
y.shape

(85271, 8979)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, SimpleRNN

embed_dim = 50
rnn_units = 128

rnn_model = Sequential([
    Embedding(vocab_size, embed_dim, input_length=max_len),
    SimpleRNN(rnn_units, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])

rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
Lstm_model = Sequential([
    Embedding(vocab_size, embed_dim, input_length=max_len),
    LSTM(rnn_units, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])

Lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
Lstm_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_rnn = rnn_model.fit(X_padded, y, epochs=10, batch_size=128, validation_split=0.1, verbose = 1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 52s 79ms/step - accuracy: 0.0287 - loss: 7.6639 - val_accuracy: 0.0376 - val_loss: 6.7677
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0429 - loss: 6.5748 - val_accuracy: 0.0509 - val_loss: 6.7198
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0360 - loss: 7.3352 - val_accuracy: 0.0360 - val_loss: 7.3177
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0460 - loss: 6.7582 - val_accuracy: 0.0442 - val_loss: 6.8990
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.0522 - loss: 6.4477 - val_accuracy: 0.0485 - val_loss: 6.8394
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0545 - loss: 6.3078 - val_accuracy: 0.0535 - val_loss: 6.8185
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 69ms/step - accuracy: 0.0580 - loss: 6.1954 - val_accuracy: 0.0510 - val_loss: 6.8116
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.0612 - loss: 6.0927 - 

In [ ]:
history_lstm = Lstm_model.fit(X_padded, y, epochs=100, batch_size=128, validation_split=0.1, verbose = 1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1557 - loss: 4.8402 - val_accuracy: 0.1148 - val_loss: 6.5504
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1618 - loss: 4.7181 - val_accuracy: 0.1114 - val_loss: 6.6099
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1684 - loss: 4.5983 - val_accuracy: 0.1156 - val_loss: 6.6773
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1757 - loss: 4.4801 - val_accuracy: 0.1150 - val_loss: 6.7461
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1847 - loss: 4.3649 - val_accuracy: 0.1147 - val_loss: 6.8108
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1958 - loss: 4.2535 - val_accuracy: 0.1157 - val_loss: 6.8770
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.2083 - loss: 4.1443 - val_accuracy: 0.1144 - val_loss: 6.9617
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.2224 - loss: 4

In [ ]:
Lstm_model.save('/content/drive/MyDrive/Colab Notebooks/model_lstm.h5')
index_to_word = {}

In [ ]:
for word, index in word_indx.items():
  index_to_word[index] = word

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from re import VERBOSE
def predictor(model, tokenizer, text_seq, max_len):
  text_seq = text_seq.lower()
  seq = tokenizer.texts_to_sequences([text_seq])
  seq = pad_sequences(seq, maxlen=max_len, padding='pre')

  pred = model.predict(seq, verbose = 0)
  pred_word_indx = np.argmax(pred)
  pred_word = index_to_word[pred_word_indx]

  return pred_word


In [ ]:
def generate_text(model, tokenizer, max_len, seed_text, n_words):
  text = []
  for _ in range(n_words):
    pred_word = predictor(model, tokenizer, seed_text, max_len)
    text.append(pred_word)
    seed_text = seed_text + ' ' + pred_word

  return ' '.join(text)

In [ ]:
seed = "The meaning of"
generate_text(Lstm_model, tokenizer, max_len, seed, 10)

'everyday life was i think about how you made them'

In [ ]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)

with open("index_to_word.pkl", "wb") as f:
  pickle.dump(index_to_word, f)